# Housing Price Prediction with Linear Regression from Scratch

This notebook implements a complete housing price prediction system using Linear Regression built from scratch with Gradient Descent. We'll use the California Housing dataset for our analysis.

## Table of Contents
1. [Data Loading and Preparation](#data-loading)
2. [Exploratory Data Analysis](#eda)
3. [Linear Regression Implementation from Scratch](#linear-regression)
4. [Model Training and Evaluation](#training)
5. [Comparison with Scikit-learn](#comparison)
6. [Practical Application](#practical)

## Learning Objectives
- Understand the mathematical foundations of Linear Regression
- Implement Gradient Descent algorithm from scratch
- Learn data preprocessing and feature engineering techniques
- Practice data visualization and exploratory data analysis
- Compare custom implementation with industry-standard libraries

## 1. Import Required Libraries

In [ ]:
# Import essential libraries for data manipulation and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

# Set random seed for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

<a id="data-loading"></a>
## 2. Data Loading and Preparation

We'll use the California Housing dataset, which contains information about housing districts in California from the 1990 census.

In [ ]:
# Load the California Housing dataset
california_housing = fetch_california_housing()

# Create a DataFrame for easier manipulation
data = pd.DataFrame(california_housing.data, columns=california_housing.feature_names)
data['target'] = california_housing.target

print("Dataset loaded successfully!")
print(f"Shape of the dataset: {data.shape}")
print(f"Features: {list(california_housing.feature_names)}")
print(f"Target: House value in hundreds of thousands of dollars")

In [ ]:
# Display basic information about the dataset
print("Dataset Info:")
print(data.info())
print("\nFirst 5 rows:")
data.head()

In [ ]:
# Statistical summary of the dataset
print("Statistical Summary:")
data.describe()

In [ ]:
# Check for missing values
print("Missing values:")
print(data.isnull().sum())

# Check for duplicates
print(f"\nDuplicate rows: {data.duplicated().sum()}")

<a id="eda"></a>
## 3. Exploratory Data Analysis (EDA)

Let's explore the data through various visualizations to understand the relationships between features and the target variable.

In [ ]:
# Distribution of the target variable (house prices)
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(data['target'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('Distribution of House Prices')
plt.xlabel('House Value (in $100k)')
plt.ylabel('Frequency')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(data['target'])
plt.title('Box Plot of House Prices')
plt.ylabel('House Value (in $100k)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Target variable statistics:")
print(f"Mean: ${data['target'].mean():.2f} (in $100k)")
print(f"Median: ${data['target'].median():.2f} (in $100k)")
print(f"Standard Deviation: ${data['target'].std():.2f} (in $100k)")

In [ ]:
# Distribution of all features
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, column in enumerate(california_housing.feature_names):
    axes[idx].hist(data[column], bins=30, alpha=0.7, color='lightcoral')
    axes[idx].set_title(f'Distribution of {column}')
    axes[idx].set_xlabel(column)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix heatmap
plt.figure(figsize=(10, 8))
correlation_matrix = data.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5)
plt.title('Correlation Matrix of Features and Target')
plt.tight_layout()
plt.show()

# Print strongest correlations with target
target_corr = correlation_matrix['target'].abs().sort_values(ascending=False)
print("Features correlation with target (absolute values):")
for feature, corr in target_corr.items():
    if feature != 'target':
        print(f"{feature}: {corr:.3f}")

In [ ]:
# Scatter plots of most correlated features with target
top_features = ['MedInc', 'AveRooms', 'AveBedrms', 'Population']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, feature in enumerate(top_features):
    axes[idx].scatter(data[feature], data['target'], alpha=0.5, s=1)
    axes[idx].set_xlabel(feature)
    axes[idx].set_ylabel('House Value (in $100k)')
    axes[idx].set_title(f'House Value vs {feature}')
    axes[idx].grid(True, alpha=0.3)
    
    # Add correlation coefficient to the plot
    corr = data[feature].corr(data['target'])
    axes[idx].text(0.05, 0.95, f'Correlation: {corr:.3f}', 
                   transform=axes[idx].transAxes, fontsize=10,
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.show()

<a id="linear-regression"></a>
## 4. Linear Regression Implementation from Scratch

Now we'll implement Linear Regression with Gradient Descent algorithm from scratch. This will help us understand the mathematical foundations of the algorithm.

In [ ]:
class LinearRegressionScratch:
    """
    Linear Regression implementation from scratch using Gradient Descent.
    
    The algorithm minimizes the cost function J(θ) = (1/2m) * Σ(hθ(x) - y)²
    where hθ(x) = θ₀ + θ₁x₁ + θ₂x₂ + ... + θₙxₙ
    """
    
    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6):
        """
        Initialize the Linear Regression model.
        
        Parameters:
        learning_rate (float): Step size for gradient descent
        max_iterations (int): Maximum number of iterations
        tolerance (float): Convergence tolerance
        """
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.tolerance = tolerance
        self.weights = None
        self.bias = None
        self.cost_history = []
        
    def _add_bias_term(self, X):
        """
        Add bias term (column of ones) to the feature matrix.
        """
        return np.column_stack([np.ones(X.shape[0]), X])
    
    def _compute_cost(self, X, y, theta):
        """
        Compute the cost function (Mean Squared Error).
        
        J(θ) = (1/2m) * Σ(hθ(x) - y)²
        """
        m = X.shape[0]
        predictions = X.dot(theta)
        cost = (1 / (2 * m)) * np.sum((predictions - y) ** 2)
        return cost
    
    def _compute_gradients(self, X, y, theta):
        """
        Compute gradients for gradient descent.
        
        ∂J/∂θⱼ = (1/m) * Σ(hθ(x) - y) * xⱼ
        """
        m = X.shape[0]
        predictions = X.dot(theta)
        gradients = (1 / m) * X.T.dot(predictions - y)
        return gradients
    
    def fit(self, X, y):
        """
        Train the model using gradient descent.
        
        Parameters:
        X (numpy.ndarray): Feature matrix
        y (numpy.ndarray): Target vector
        """
        # Add bias term to X
        X_with_bias = self._add_bias_term(X)
        
        # Initialize parameters (weights + bias)
        n_features = X_with_bias.shape[1]
        theta = np.random.normal(0, 0.01, n_features)
        
        # Gradient descent
        prev_cost = float('inf')
        
        for iteration in range(self.max_iterations):
            # Compute cost
            cost = self._compute_cost(X_with_bias, y, theta)
            self.cost_history.append(cost)
            
            # Check for convergence
            if abs(prev_cost - cost) < self.tolerance:
                print(f"Converged at iteration {iteration + 1}")
                break
                
            # Compute gradients
            gradients = self._compute_gradients(X_with_bias, y, theta)
            
            # Update parameters
            theta = theta - self.learning_rate * gradients
            prev_cost = cost
        
        # Store final parameters
        self.bias = theta[0]
        self.weights = theta[1:]
        
        print(f"Training completed! Final cost: {self.cost_history[-1]:.6f}")
    
    def predict(self, X):
        """
        Make predictions using the trained model.
        
        Parameters:
        X (numpy.ndarray): Feature matrix
        
        Returns:
        numpy.ndarray: Predictions
        """
        if self.weights is None or self.bias is None:
            raise ValueError("Model must be trained before making predictions!")
        
        return X.dot(self.weights) + self.bias
    
    def get_parameters(self):
        """
        Get the learned parameters.
        
        Returns:
        tuple: (weights, bias)
        """
        return self.weights, self.bias

print("Linear Regression class implemented successfully!")

## 5. Data Preprocessing

Before training our model, we need to preprocess the data by splitting it into training and testing sets and scaling the features.

In [ ]:
# Prepare features and target
X = data.drop('target', axis=1).values
y = data['target'].values

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set - Features: {X_train.shape}, Target: {y_train.shape}")
print(f"Testing set - Features: {X_test.shape}, Target: {y_test.shape}")

In [ ]:
# Feature scaling using StandardScaler
# This is crucial for gradient descent to converge properly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")
print(f"Original training data range:")
print(f"Min: {X_train.min(axis=0)}")
print(f"Max: {X_train.max(axis=0)}")
print(f"\nScaled training data range:")
print(f"Min: {X_train_scaled.min(axis=0).round(3)}")
print(f"Max: {X_train_scaled.max(axis=0).round(3)}")
print(f"Mean: {X_train_scaled.mean(axis=0).round(3)}")
print(f"Std: {X_train_scaled.std(axis=0).round(3)}")

<a id="training"></a>
## 6. Model Training and Evaluation

Now let's train our custom Linear Regression model and evaluate its performance.

In [ ]:
# Initialize and train our custom Linear Regression model
model_scratch = LinearRegressionScratch(
    learning_rate=0.01, 
    max_iterations=1000, 
    tolerance=1e-6
)

print("Training our custom Linear Regression model...")
model_scratch.fit(X_train_scaled, y_train)

In [ ]:
# Plot the cost function over iterations
plt.figure(figsize=(10, 6))
plt.plot(model_scratch.cost_history, linewidth=2, color='red')
plt.title('Cost Function Over Iterations')
plt.xlabel('Iteration')
plt.ylabel('Cost (Mean Squared Error)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Initial cost: {model_scratch.cost_history[0]:.6f}")
print(f"Final cost: {model_scratch.cost_history[-1]:.6f}")
print(f"Cost reduction: {model_scratch.cost_history[0] - model_scratch.cost_history[-1]:.6f}")

In [ ]:
# Make predictions on both training and test sets
y_train_pred = model_scratch.predict(X_train_scaled)
y_test_pred = model_scratch.predict(X_test_scaled)

print("Predictions completed!")
print(f"Training predictions shape: {y_train_pred.shape}")
print(f"Test predictions shape: {y_test_pred.shape}")

In [ ]:
# Define evaluation metrics
def mean_squared_error_custom(y_true, y_pred):
    """Calculate Mean Squared Error"""
    return np.mean((y_true - y_pred) ** 2)

def root_mean_squared_error(y_true, y_pred):
    """Calculate Root Mean Squared Error"""
    return np.sqrt(mean_squared_error_custom(y_true, y_pred))

def r2_score_custom(y_true, y_pred):
    """Calculate R-squared (coefficient of determination)"""
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

def mean_absolute_error(y_true, y_pred):
    """Calculate Mean Absolute Error"""
    return np.mean(np.abs(y_true - y_pred))

# Calculate metrics for our custom model
train_mse = mean_squared_error_custom(y_train, y_train_pred)
test_mse = mean_squared_error_custom(y_test, y_test_pred)
train_rmse = root_mean_squared_error(y_train, y_train_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score_custom(y_train, y_train_pred)
test_r2 = r2_score_custom(y_test, y_test_pred)
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("\n" + "="*50)
print("CUSTOM LINEAR REGRESSION MODEL EVALUATION")
print("="*50)
print(f"Training Set Metrics:")
print(f"  MSE:  {train_mse:.6f}")
print(f"  RMSE: {train_rmse:.6f}")
print(f"  R²:   {train_r2:.6f}")
print(f"  MAE:  {train_mae:.6f}")
print(f"\nTest Set Metrics:")
print(f"  MSE:  {test_mse:.6f}")
print(f"  RMSE: {test_rmse:.6f}")
print(f"  R²:   {test_r2:.6f}")
print(f"  MAE:  {test_mae:.6f}")
print("="*50)

In [ ]:
# Visualize predictions vs actual values
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Training set
axes[0].scatter(y_train, y_train_pred, alpha=0.5, s=10)
axes[0].plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title(f'Training Set: Actual vs Predicted\nR² = {train_r2:.3f}')
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, s=10, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')
axes[1].set_title(f'Test Set: Actual vs Predicted\nR² = {test_r2:.3f}')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze residuals (prediction errors)
train_residuals = y_train - y_train_pred
test_residuals = y_test - y_test_pred

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Training residuals vs predicted
axes[0, 0].scatter(y_train_pred, train_residuals, alpha=0.5, s=10)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Predicted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Training Set: Residuals vs Predicted')
axes[0, 0].grid(True, alpha=0.3)

# Test residuals vs predicted
axes[0, 1].scatter(y_test_pred, test_residuals, alpha=0.5, s=10, color='orange')
axes[0, 1].axhline(y=0, color='r', linestyle='--')
axes[0, 1].set_xlabel('Predicted Values')
axes[0, 1].set_ylabel('Residuals')
axes[0, 1].set_title('Test Set: Residuals vs Predicted')
axes[0, 1].grid(True, alpha=0.3)

# Training residuals distribution
axes[1, 0].hist(train_residuals, bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Training Set: Residuals Distribution')
axes[1, 0].grid(True, alpha=0.3)

# Test residuals distribution
axes[1, 1].hist(test_residuals, bins=50, alpha=0.7, color='lightcoral', edgecolor='black')
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Test Set: Residuals Distribution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Training residuals - Mean: {train_residuals.mean():.6f}, Std: {train_residuals.std():.6f}")
print(f"Test residuals - Mean: {test_residuals.mean():.6f}, Std: {test_residuals.std():.6f}")

<a id="comparison"></a>
## 7. Comparison with Scikit-learn

Let's compare our custom implementation with the industry-standard scikit-learn LinearRegression.

In [ ]:
# Train scikit-learn Linear Regression model
sklearn_model = LinearRegression()
sklearn_model.fit(X_train_scaled, y_train)

# Make predictions
sklearn_train_pred = sklearn_model.predict(X_train_scaled)
sklearn_test_pred = sklearn_model.predict(X_test_scaled)

# Calculate metrics
sklearn_train_mse = mean_squared_error(y_train, sklearn_train_pred)
sklearn_test_mse = mean_squared_error(y_test, sklearn_test_pred)
sklearn_train_r2 = r2_score(y_train, sklearn_train_pred)
sklearn_test_r2 = r2_score(y_test, sklearn_test_pred)

print("\n" + "="*60)
print("MODEL COMPARISON: CUSTOM vs SCIKIT-LEARN")
print("="*60)
print(f"{'Metric':<15} {'Custom (Train)':<15} {'Custom (Test)':<15} {'SKLearn (Train)':<16} {'SKLearn (Test)':<15}")
print("-"*90)
print(f"{'MSE':<15} {train_mse:<15.6f} {test_mse:<15.6f} {sklearn_train_mse:<16.6f} {sklearn_test_mse:<15.6f}")
print(f"{'R²':<15} {train_r2:<15.6f} {test_r2:<15.6f} {sklearn_train_r2:<16.6f} {sklearn_test_r2:<15.6f}")
print("="*60)

# Compare parameters
custom_weights, custom_bias = model_scratch.get_parameters()
sklearn_weights = sklearn_model.coef_
sklearn_bias = sklearn_model.intercept_

print("\nParameter Comparison:")
print(f"Bias - Custom: {custom_bias:.6f}, Scikit-learn: {sklearn_bias:.6f}")
print("\nWeights comparison:")
feature_names = california_housing.feature_names
for i, feature in enumerate(feature_names):
    print(f"{feature:<12} - Custom: {custom_weights[i]:.6f}, Scikit-learn: {sklearn_weights[i]:.6f}")

In [ ]:
# Visual comparison of predictions
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Compare predictions on test set
axes[0].scatter(y_test_pred, sklearn_test_pred, alpha=0.6, s=15)
min_val = min(y_test_pred.min(), sklearn_test_pred.min())
max_val = max(y_test_pred.max(), sklearn_test_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[0].set_xlabel('Custom Model Predictions')
axes[0].set_ylabel('Scikit-learn Predictions')
axes[0].set_title('Prediction Comparison: Custom vs Scikit-learn')
axes[0].grid(True, alpha=0.3)

# Difference in predictions
pred_diff = y_test_pred - sklearn_test_pred
axes[1].hist(pred_diff, bins=50, alpha=0.7, color='green', edgecolor='black')
axes[1].set_xlabel('Prediction Difference (Custom - Scikit-learn)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Prediction Differences')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Prediction difference statistics:")
print(f"Mean difference: {pred_diff.mean():.8f}")
print(f"Std of difference: {pred_diff.std():.8f}")
print(f"Max absolute difference: {np.abs(pred_diff).max():.8f}")

<a id="practical"></a>
## 8. Practical Application: Predicting House Prices

Let's use our trained model to predict house prices for new examples and demonstrate its practical application.

In [ ]:
# Create sample houses for prediction
sample_houses = {
    'Luxury House': [15.0, 8.0, 1.2, 3000, 2.5, 35.5, -120.3, 500000],  # High income area
    'Average House': [5.0, 6.0, 1.0, 3000, 3.0, 34.0, -118.5, 200000],   # Medium income area
    'Budget House': [2.5, 4.5, 1.1, 4000, 4.5, 32.8, -117.2, 100000],    # Lower income area
    'Beachfront House': [12.0, 7.5, 1.0, 2000, 2.0, 33.8, -118.4, 800000], # Near coast
    'Suburban House': [7.2, 6.2, 1.05, 2500, 3.2, 34.2, -117.8, 350000]   # Suburban area
}

print("Sample Houses for Prediction:")
print("=" * 70)
print(f"{'House Type':<18} {'Features':<50}")
print("-" * 70)

feature_names_full = list(california_housing.feature_names)
sample_features = []

for house_name, features in sample_houses.items():
    # Extract only the 8 features (excluding population which is the 8th index)
    house_features = features[:8]  # All 8 features
    sample_features.append(house_features)
    
    feature_str = ", ".join([f"{name}: {val}" for name, val in zip(feature_names_full, house_features)])
    print(f"{house_name:<18} {feature_str[:47]}{'...' if len(feature_str) > 47 else ''}")

sample_features = np.array(sample_features)
print(f"\nSample features shape: {sample_features.shape}")

In [ ]:
# Scale the sample features using the same scaler
sample_features_scaled = scaler.transform(sample_features)

# Make predictions using both models
custom_predictions = model_scratch.predict(sample_features_scaled)
sklearn_predictions = sklearn_model.predict(sample_features_scaled)

print("\n" + "="*80)
print("HOUSE PRICE PREDICTIONS")
print("="*80)
print(f"{'House Type':<18} {'Custom Model':<15} {'Scikit-learn':<15} {'Difference':<12}")
print("-"*80)

for i, (house_name, _) in enumerate(sample_houses.items()):
    custom_price = custom_predictions[i] * 100  # Convert back to thousands
    sklearn_price = sklearn_predictions[i] * 100
    difference = abs(custom_price - sklearn_price)
    
    print(f"{house_name:<18} ${custom_price:<14,.0f} ${sklearn_price:<14,.0f} ${difference:<11,.0f}")

print("="*80)
print("Note: Prices are in thousands of dollars (multiply by 1000 for actual price)")

In [ ]:
# Visualize feature importance (absolute weights)
feature_importance = np.abs(custom_weights)
sorted_indices = np.argsort(feature_importance)[::-1]

plt.figure(figsize=(12, 8))
bars = plt.bar(range(len(feature_importance)), feature_importance[sorted_indices], 
               color='skyblue', edgecolor='navy', alpha=0.7)
plt.xlabel('Features')
plt.ylabel('Absolute Weight (Feature Importance)')
plt.title('Feature Importance in Our Custom Linear Regression Model')
plt.xticks(range(len(feature_importance)), 
           [feature_names[i] for i in sorted_indices], rotation=45)
plt.grid(True, alpha=0.3)

# Add value labels on bars
for i, bar in enumerate(bars):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.3f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

print("\nFeature Importance Ranking:")
for i, idx in enumerate(sorted_indices):
    print(f"{i+1}. {feature_names[idx]}: {feature_importance[idx]:.6f}")

In [ ]:
# Interactive prediction function
def predict_house_price(med_inc, house_age, ave_rooms, ave_bedrms, population, ave_occup, latitude, longitude):
    """
    Predict house price for given features.
    
    Parameters:
    med_inc: Median income in block group
    house_age: Median house age in block group
    ave_rooms: Average number of rooms per household
    ave_bedrms: Average number of bedrooms per household
    population: Block group population
    ave_occup: Average number of household members
    latitude: Block group latitude
    longitude: Block group longitude
    
    Returns:
    Predicted house price in hundreds of thousands of dollars
    """
    # Create feature array
    features = np.array([[med_inc, house_age, ave_rooms, ave_bedrms, population, ave_occup, latitude, longitude]])
    
    # Scale features
    features_scaled = scaler.transform(features)
    
    # Make prediction
    prediction = model_scratch.predict(features_scaled)[0]
    
    return prediction

# Example usage
print("\n" + "="*60)
print("INTERACTIVE HOUSE PRICE PREDICTION FUNCTION")
print("="*60)

# Test the function with a custom house
example_price = predict_house_price(
    med_inc=8.0,      # $80,000 median income
    house_age=10.0,   # 10 years old
    ave_rooms=7.0,    # 7 rooms per household
    ave_bedrms=1.0,   # 1 bedroom per household
    population=3000,  # 3000 people in block
    ave_occup=3.0,    # 3 people per household
    latitude=34.0,    # Los Angeles area
    longitude=-118.0  # Los Angeles area
)

print(f"\nExample prediction:")
print(f"For a house with the above characteristics:")
print(f"Predicted price: ${example_price:.2f} (in hundreds of thousands)")
print(f"Actual price estimate: ${example_price * 100:.0f},000")

print("\nYou can use this function to predict prices for any house by providing the 8 features!")

## 9. Summary and Key Learnings

### What We Accomplished:

1. **📊 Data Analysis**: We thoroughly explored the California Housing dataset with visualizations including histograms, correlation heatmaps, and scatter plots.

2. **🔧 Linear Regression from Scratch**: We implemented a complete Linear Regression algorithm using Gradient Descent, understanding the mathematical foundations:
   - Cost function: J(θ) = (1/2m) * Σ(hθ(x) - y)²
   - Gradient computation: ∂J/∂θⱼ = (1/m) * Σ(hθ(x) - y) * xⱼ
   - Parameter updates: θⱼ := θⱼ - α * ∂J/∂θⱼ

3. **📈 Model Evaluation**: We used multiple metrics to evaluate performance:
   - Mean Squared Error (MSE)
   - Root Mean Squared Error (RMSE)
   - R-squared (R²)
   - Mean Absolute Error (MAE)

4. **🔍 Comparison**: We validated our implementation against scikit-learn's LinearRegression and found nearly identical results.

5. **🏠 Practical Application**: We created a prediction function that can estimate house prices for new properties.

### Key Insights:
- **Most Important Features**: Median Income (MedInc) is the strongest predictor of house prices
- **Model Performance**: Our custom implementation achieved similar performance to scikit-learn
- **Feature Scaling**: Critical for gradient descent convergence
- **Convergence**: The cost function decreased smoothly, indicating proper implementation

### Machine Learning Concepts Learned:
- **Gradient Descent**: Iterative optimization algorithm
- **Feature Scaling**: Standardization for better convergence
- **Overfitting/Underfitting**: Model evaluation on train vs test sets
- **Residual Analysis**: Understanding prediction errors
- **Feature Engineering**: Importance of different features

This implementation demonstrates the fundamental concepts of machine learning and provides a solid foundation for understanding more complex algorithms!